# 🌍 Country Development Clustering - Exploratory Data Analysis (EDA)

## 📋 Notebook Purpose

This notebook focuses on **understanding the data** before clustering:
1. **Explore** country development indicators across 168 countries
2. **Analyze distributions** of economic, social, and health features
3. **Identify outliers and data quality issues**
4. **Transform skewed features** using Yeo-Johnson transformation
5. **Visualize relationships** between development indicators

## 📊 Dataset Overview

| Aspect | Detail |
|:---|:---|
| **Source** | Country development indicators dataset |
| **Countries** | 168 countries |
| **Features** | 9 development indicators |
| **Goal** | Prepare data for clustering |

### 🔍 Features Analyzed

| Feature | Description | Type |
|:---|:---|:---|
| `child_mort` | Child mortality rate (per 1000 births) | Health |
| `exports` | Exports as % of GDP | Economic |
| `health` | Health expenditure per capita | Health |
| `imports` | Imports as % of GDP | Economic |
| `income` | Net income per person | Economic |
| `inflation` | Annual inflation rate (%) | Economic |
| `life_expec` | Life expectancy at birth (years) | Health |
| `total_fer` | Total fertility rate (births per woman) | Social |
| `gdpp` | GDP per capita | Economic |

## 📁 Notebook Structure
```
01-eda-analysis.ipynb
├── 1. Data Loading & Initial Inspection
│ ├── Load raw data
│ ├── Check missing values
│ └── View data structure
├── 2. Exploratory Data Analysis
│ ├── Distribution analysis (histograms)
│ ├── Correlation analysis (heatmap)
│ ├── Outlier detection (boxplots)
│ ├── Multivariate outlier analysis
│ └── Skewness & kurtosis analysis
├── 3. Data Transformation
│ ├── Yeo-Johnson transformation
│ ├── Standardization
│ └── Transformation impact analysis
├── 4. Data Quality Summary
│ ├── Before/after comparison
│ └── Export cleaned data
└── 5. Next Steps
└── Ready for clustering
```


## 🔧 Key Findings from EDA

- **Skewed features**: Inflation (5.15), Exports (2.45), Income (2.23)
- **Outliers**: Nigeria (inflation: 104%), Luxembourg (high income/GDP)
- **Transformation**: Yeo-Johnson reduced skewness to < 0.2 for all features
- **Data quality**: No missing values, 168 countries, 9 features

## 📤 Output

- Cleaned data: `../data/processed/countries_transformed_scaled.csv`
- EDA figures: `../figures/eda/`
- Ready for: `02-clustering-analysis.ipynb`



In [ ]:
# ============================================================================
# 1. SETUP AND IMPORTS
# ============================================================================

import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine learning
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import PowerTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer




# Statistical tests
from scipy.stats import f_oneway, kruskal
import scipy.stats as stats



# Get paths
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)




print("✅ All imports successful!")
print("=" * 50)

In [ ]:
# ============================================================================
# 2. LOAD AND EXPLORE DATA
# ============================================================================

# Load data
data_path = "../data/raw/Country-data.csv"
df_raw = pd.read_csv(data_path)

print(f"✅ Data loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print("=" * 60)

# Display first few rows
print("\n📊 FIRST 5 ROWS OF THE DATASET:")
display(df_raw.head())

# Display last few rows
print("\n📊 LAST 5 ROWS OF THE DATASET:")
display(df_raw.tail())

In [ ]:
# ============================================================================
# 2.2 Data Quality Check
# ============================================================================

print("🔍 DATA QUALITY CHECK")
print("=" * 60)

# Missing values analysis
missing_values = df_raw.isnull().sum()
missing_percent = (missing_values / len(df_raw) * 100)

missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Values': missing_values.values,
    'Missing %': missing_percent.values
})
missing_df = missing_df[missing_df['Missing Values'] > 0].sort_values('Missing %', ascending=False)

if len(missing_df) > 0:
    print("📊 COLUMNS WITH MISSING VALUES:")
    display(missing_df)
else:
    print("✅ No missing values found in the dataset!")

# Duplicate check
duplicates = df_raw.duplicated().sum()
print(f"\n🔢 Duplicate rows: {duplicates:,}")

# Complete rows
complete_rows = len(df_raw) - df_raw.isnull().any(axis=1).sum()
print(f"✅ Complete rows: {complete_rows:,} ({complete_rows/len(df_raw)*100:.1f}%)")

In [ ]:
# ============================================================================
# 2.3 Statistical Summary
# ============================================================================

print("📊 STATISTICAL SUMMARY")
print("=" * 60)

# Get only numeric columns
numeric_df = df_raw.select_dtypes(include=[np.number])

if not numeric_df.empty:
    print("DESCRIPTIVE STATISTICS FOR NUMERIC COLUMNS:")
    display(numeric_df.describe().round(2))
    
    # Additional statistics
    print("\n📈 Additional Statistics:")
    stats_df = pd.DataFrame({
        'Skewness': numeric_df.skew().round(2),
        'Kurtosis': numeric_df.kurtosis().round(2),
        'Variance': numeric_df.var().round(2),
        'Range': (numeric_df.max() - numeric_df.min()).round(2)
    })
    display(stats_df)

In [ ]:
# ============================================================================
# 2.4 Feature Selection
# ============================================================================

print("🎯 FEATURE SELECTION")
print("=" * 60)

# Identify columns to exclude
exclude_cols = ['country']

# Get numeric columns excluding identifiers
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
selected_features = [col for col in numeric_cols if col not in exclude_cols]

print(f"Total numeric columns: {len(numeric_cols)}")
print(f"Selected features for clustering: {len(selected_features)}")
print("\nSelected features:")
print("-" * 60)
for i, feature in enumerate(selected_features, 1):
    print(f"{i:2d}. {feature}")


In [ ]:
# ============================================================================
# 3. EXPLORATORY DATA ANALYSIS (EDA) - HISTOGRAMS
# ============================================================================

print("📈 EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# Select key features for visualization
key_features = selected_features[:6]

# Professional color palette
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E', '#BC4B51']
mean_color = '#D62828'
median_color = '#2C3E50'


fig, axes = plt.subplots(2, 3, figsize=(18, 12))  
axes = axes.flatten()

for i, feature in enumerate(key_features):
    # Create histogram
    n, bins, patches = axes[i].hist(
        df_raw[feature].dropna(), 
        bins=30, 
        edgecolor='white', 
        alpha=0.8,
        color=colors[i % len(colors)],
        linewidth=1.2
    )
    
    # Add mean and median lines
    mean_val = df_raw[feature].mean()
    median_val = df_raw[feature].median()
    
    axes[i].axvline(mean_val, color=mean_color, linestyle='--', 
                    linewidth=2.5, alpha=0.9,
                    label=f'Mean: {mean_val:.2f}')
    axes[i].axvline(median_val, color=median_color, linestyle='-', 
                    linewidth=2.0, alpha=0.8,
                    label=f'Median: {median_val:.2f}')
    
 
    axes[i].set_title(f'Distribution of {feature}', 
                      fontsize=22, fontweight='bold', pad=15)
    axes[i].set_xlabel(feature, fontsize=20, fontweight='medium')
    axes[i].set_ylabel('Frequency', fontsize=20, fontweight='medium')
    
   
    axes[i].tick_params(axis='both', labelsize=18, width=1.5, length=6)
    
    # ====== LEGEND ======
    axes[i].legend(loc='upper right', 
                   framealpha=0.9, 
                   edgecolor='#CCCCCC',
                   fontsize=18)
    
    # ====== GRID AND STYLING ======
    axes[i].grid(True, alpha=0.2, linestyle='-', linewidth=0.5)
    axes[i].set_axisbelow(True)
    axes[i].set_facecolor('#F8F9FA')
    
    # ====== SPINES ======
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)
    axes[i].spines['left'].set_color('#333333')
    axes[i].spines['bottom'].set_color('#333333')
    axes[i].spines['left'].set_linewidth(1.5)
    axes[i].spines['bottom'].set_linewidth(1.5)

# Main title
plt.suptitle('Distribution of Key Development Indicators', 
             fontsize=24, 
             fontweight='bold',
             y=1.02,
             color='#1a1a1a')

plt.tight_layout()

# ====== SAVE WITH HIGHER DPI ======
plt.savefig('../figures/eda/key_variable_distributions.png', 
            dpi=300, 
            bbox_inches='tight',
            facecolor='white',
            edgecolor='none')

plt.show()

print("✅ Enhanced distribution plots saved to: ../figures/eda/key_variable_distributions.png")

In [ ]:
# ============================================================================
# 3.1 Correlation Analysis - PDF OPTIMIZED
# ============================================================================

# Calculate correlation matrix
corr_matrix = df_raw[selected_features].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))


fig, ax = plt.subplots(figsize=(18, 14)) 

cmap = 'PRGn'


heatmap = sns.heatmap(corr_matrix, 
                      mask=mask,
                      annot=True, 
                      fmt='.2f',
                      cmap=cmap,
                      center=0,
                      square=True,
                      linewidths=1.5,
                      linecolor='white',
                      cbar_kws={
                          "shrink": 0.8,
                          "aspect": 30,
                          "label": "Correlation Coefficient",
                          "ticks": [-1, -0.5, 0, 0.5, 1]
                      },
                      annot_kws={
                          "size": 18,      
                          "weight": "bold",
                          "color": "black"
                      },
                      vmin=-1, vmax=1)

cbar = heatmap.collections[0].colorbar
cbar.ax.tick_params(labelsize=18, width=1.5)  
cbar.set_label('Correlation Coefficient', 
               fontsize=20,        
               weight='bold')

# ====== MAIN TITLE ======
plt.title('Correlation Matrix of Development Indicators', 
          fontsize=26,            
          fontweight='bold', 
          pad=25,
          color='#1a1a1a')


plt.xticks(rotation=45, ha='right', fontsize=18)  
plt.yticks(rotation=0, fontsize=18)               

# Remove the default axis labels
plt.xlabel('', fontsize=0)
plt.ylabel('', fontsize=0)

# ====== ADD SUBTLE BACKGROUND ======
ax.set_facecolor('#f8f9fa')

# ====== ADJUST LAYOUT ======
plt.tight_layout()

# ====== SAVE WITH HIGHER DPI ======
plt.savefig('../figures/eda/correlation_matrix.png', 
            dpi=300, 
            bbox_inches='tight',
            facecolor='white',
            edgecolor='none')

plt.show()

# Find highly correlated features
print("\n📊 HIGHLY CORRELATED FEATURES (>0.8):")
print("-" * 60)
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], 
                            corr_matrix.iloc[i, j]))

if high_corr:
    for feature1, feature2, corr in high_corr:
        print(f"   {feature1} ↔ {feature2}: {corr:.3f}")
else:
    print("   No highly correlated features found (>0.8)")

print("\n✅ Correlation matrix saved to: ../figures/eda/correlation_matrix.png")

In [ ]:
# ============================================================================
# 3.2 Outlier Detection - Boxplot Visualization 
# ============================================================================

print("📊 OUTLIER DETECTION VISUALIZATION")
print("=" * 60)


fig, axes = plt.subplots(2, 1, figsize=(20, 16))  

# ============================================================================
# Plot 1: All features with outliers highlighted
# ============================================================================

ax1 = axes[0]
features_to_plot = selected_features

# Create boxplot with outlier highlighting 
boxplot = ax1.boxplot(df_raw[features_to_plot].values, 
                       tick_labels=features_to_plot,  
                       patch_artist=True,
                       showmeans=True,
                       meanline=True,
                       meanprops={'color': 'red', 'linestyle': '--', 'linewidth': 2.5},
                       medianprops={'color': 'darkblue', 'linewidth': 2.5},
                       whiskerprops={'color': 'gray', 'linewidth': 1.5},
                       capprops={'color': 'gray', 'linewidth': 1.5},
                       flierprops={'marker': 'o', 'markerfacecolor': 'red', 
                                  'markersize': 12, 'alpha': 0.6}) 

# Color the boxes
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E', 
          '#BC4B51', '#5D4E6D', '#E59866', '#48C9B0']
for patch, color in zip(boxplot['boxes'], colors[:len(features_to_plot)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add outlier counts with proper positioning 
for i, feature in enumerate(features_to_plot):
    # Calculate outliers using IQR
    Q1 = df_raw[feature].quantile(0.25)
    Q3 = df_raw[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df_raw[(df_raw[feature] < lower_bound) | (df_raw[feature] > upper_bound)]
    
    # Add count annotation with proper spacing
    if len(outliers) > 0:
        
        max_val = df_raw[feature].max()
        y_pos = max_val * 1.08  
        
        ax1.text(i + 1, y_pos, 
                f'n={len(outliers)}', 
                ha='center', va='bottom',
                fontsize=18, 
                fontweight='bold',
                color='red',
                bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.9, edgecolor='red'))

ax1.set_title('Outlier Detection in Development Indicators\n(Red dots = outliers, Dashed red line = mean)', 
             fontsize=24, fontweight='bold', pad=20)  
ax1.set_ylabel('Value', fontsize=20, fontweight='bold') 
ax1.tick_params(axis='y', labelsize=18)  
ax1.tick_params(axis='x', labelsize=18)  
ax1.grid(True, alpha=0.2, axis='y')
ax1.set_axisbelow(True)

# X-axis labels
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right', fontsize=18, fontweight='bold') 

# Add padding to y-axis to accommodate labels
ymin, ymax = ax1.get_ylim()
ax1.set_ylim(ymin, ymax * 1.12)  

# ============================================================================
# Plot 2: Log-transformed view of skewed features
# ============================================================================

ax2 = axes[1]

# Identify skewed features
skewed_features = ['income', 'gdpp', 'inflation', 'exports', 'imports', 'child_mort']
skewed_data = df_raw[skewed_features].copy()

# Apply log transform for visualization
for feature in skewed_features:
    if feature == 'inflation':
        skewed_data[feature] = np.log1p(skewed_data[feature] + abs(skewed_data[feature].min()) + 1)
    else:
        skewed_data[feature] = np.log1p(skewed_data[feature])

# Create boxplot on log scale 
boxplot2 = ax2.boxplot(skewed_data.values,
                        tick_labels=skewed_features,  
                        patch_artist=True,
                        showmeans=True,
                        meanline=True,
                        meanprops={'color': 'red', 'linestyle': '--', 'linewidth': 2.5},
                        medianprops={'color': 'darkblue', 'linewidth': 2.5},
                        flierprops={'marker': 'o', 'markerfacecolor': 'red', 
                                   'markersize': 12, 'alpha': 0.6}) 

# Color the boxes
colors2 = ['#C0392B', '#E74C3C', '#F39C12', '#2980B9', '#8E44AD', '#27AE60']
for patch, color in zip(boxplot2['boxes'], colors2):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add outlier counts for log-transformed plot
for i, feature in enumerate(skewed_features):
    Q1 = skewed_data[feature].quantile(0.25)
    Q3 = skewed_data[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = skewed_data[(skewed_data[feature] < lower_bound) | (skewed_data[feature] > upper_bound)]
    
    if len(outliers) > 0:
        max_val = skewed_data[feature].max()
        y_pos = max_val * 1.08
        ax2.text(i + 1, y_pos, 
                f'n={len(outliers)}', 
                ha='center', va='bottom',
                fontsize=18,  
                fontweight='bold',
                color='red',
                bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.9, edgecolor='red'))


ax2.set_title('Log-Transformed View of Skewed Features\n(Highlights extreme outliers in income, GDP, and inflation)', 
             fontsize=24, fontweight='bold', pad=20)  
ax2.set_ylabel('Log(Value)', fontsize=20, fontweight='bold')  
ax2.tick_params(axis='y', labelsize=18)  
ax2.tick_params(axis='x', labelsize=18)  
ax2.grid(True, alpha=0.2, axis='y')
ax2.set_axisbelow(True)

# X-axis labels 
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=18, fontweight='bold') 

# Add padding to y-axis for log plot
ymin2, ymax2 = ax2.get_ylim()
ax2.set_ylim(ymin2, ymax2 * 1.12)

plt.tight_layout()
plt.savefig('../figures/eda/outlier_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Outlier boxplots saved to: ../figures/eda/outlier_boxplots.png")

In [ ]:
# ============================================================================
# 3.3  Pairplot with Log-Transformed Data 
# ============================================================================

print("📊 MULTIVARIATE OUTLIER DETECTION (Log Scale)")
print("=" * 60)



# Select top 5 features with most outliers
outlier_features = ['income', 'gdpp', 'inflation', 'exports', 'child_mort']

# Create log-transformed version for visualization
df_log = df_raw[outlier_features].copy()

for feature in outlier_features:
    if feature == 'inflation':
        # Handle negative inflation values
        df_log[feature] = np.log1p(df_log[feature] + abs(df_log[feature].min()) + 1)
    else:
        df_log[feature] = np.log1p(df_log[feature])

# Scale for Isolation Forest
scaler = StandardScaler()
df_scaled_for_outliers = pd.DataFrame(
    scaler.fit_transform(df_log),
    columns=outlier_features,
    index=df_raw.index
)

# Detect outliers
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outlier_pred = iso_forest.fit_predict(df_scaled_for_outliers)
outlier_mask = outlier_pred == -1

# Create plot data
df_plot = df_log.copy()
df_plot['Outlier'] = outlier_mask
df_plot['country'] = df_raw['country']

g = sns.pairplot(
    df_plot,
    vars=outlier_features,
    hue='Outlier',
    palette={True: 'red', False: 'blue'},
    plot_kws={'alpha': 0.6, 's': 60},  
    diag_kind='hist',
    diag_kws={'alpha': 0.7, 'bins': 30},
    corner=False,
    height=4.0  
)

# ====== RENAME AXES TO SHOW LOG-TRANSFORMED ======
for ax in g.axes.flatten():
    if ax is not None:
        # Get current label
        label = ax.get_xlabel()
        if label and label in outlier_features:
            ax.set_xlabel(f'Log({label})', fontsize=20, fontweight='medium')  
        label = ax.get_ylabel()
        if label and label in outlier_features:
            ax.set_ylabel(f'Log({label})', fontsize=20, fontweight='medium')  
        # Update tick labels
        ax.tick_params(axis='both', labelsize=18)  
        # Update subplot titles (diagonal)
        if ax.get_title():
            ax.set_title(ax.get_title(), fontsize=20, fontweight='bold')  

g.fig.suptitle('Multivariate Outlier Detection (Log Scale)\n(Red points flagged by Isolation Forest)', 
              fontsize=26, fontweight='bold', y=1.02) 


legend = g._legend
if legend:
    legend.set_title('Outlier', prop={'size': 20, 'weight': 'bold'})  
    for text in legend.get_texts():
        text.set_fontsize(20)  

# ====== ADJUST LAYOUT ======
plt.tight_layout()
plt.savefig('../figures/eda/multivariate_outliers_log.png', dpi=300, bbox_inches='tight')
plt.show()

# Print results
print("\n🚨 MULTIVARIATE OUTLIERS DETECTED:")
print("-" * 60)
outlier_countries = df_raw[outlier_mask]['country'].tolist()
for country in outlier_countries:
    print(f"   • {country}")

# Show their values
print("\n📋 OUTLIER COUNTRY PROFILES:")
print("-" * 60)
for idx in np.where(outlier_mask)[0]:
    country = df_raw.iloc[idx]['country']
    values = df_raw.iloc[idx][outlier_features].round(2)
    print(f"\n  {country}:")
    for feat, val in values.items():
        print(f"    • {feat}: {val:.2f}")

In [ ]:
# ============================================================================
# 3.4 Outlier Heatmap Table
# ============================================================================

print("📊 CREATING OUTLIER HEATMAP")
print("=" * 60)


# ============================================================================
# STEP 1: Identify Outliers
# ============================================================================

outlier_features = selected_features 

# Calculate Z-scores for all rows
z_scores_all = df_raw[outlier_features].apply(lambda x: np.abs((x - x.mean()) / x.std()))

# Filter rows where ANY feature has a Z-score > 3 (Extreme outlier)
extreme_mask = (z_scores_all > 3).any(axis=1)
outlier_df = df_raw[extreme_mask].copy()

print(f"Found {len(outlier_df)} countries with extreme outliers (Z > 3)")

# ============================================================================
# STEP 2: ENSURE COUNTRY COLUMN EXISTS
# ============================================================================

if 'country' in outlier_df.columns:
    outlier_df = outlier_df.rename(columns={'country': 'Country'})
elif 'Country' not in outlier_df.columns:
    outlier_df = outlier_df.reset_index()
    if 'index' in outlier_df.columns:
        outlier_df = outlier_df.rename(columns={'index': 'Country'})
    else:
        outlier_df['Country'] = outlier_df.index

print(f"\n📋 Countries in outlier heatmap:")
for i, country in enumerate(outlier_df['Country'].values):
    print(f"   {i+1}. {country}")

# ============================================================================
# STEP 3: Prepare Data for Heatmap
# ============================================================================

heatmap_data = outlier_df[outlier_features].copy()

# Normalize each column for heatmap (Z-scores)
for feature in outlier_features:
    mean = df_raw[feature].mean()
    std = df_raw[feature].std()
    heatmap_data[feature] = (heatmap_data[feature] - mean) / std

# ============================================================================
# STEP 4: CREATE ANNOTATION TABLE WITH COMMA FORMATTING
# ============================================================================

# Create a copy of original values for annotation
annot_data = outlier_df[outlier_features].copy()

# Format numbers with commas for thousands
def format_with_commas(value):
    """Format numbers with commas for thousands separators"""
    if pd.isna(value):
        return ''
    if isinstance(value, (int, float)):
        if abs(value) >= 1000:
            return f'{value:,.0f}'  # No decimals for large numbers
        elif abs(value) < 1 and value != 0:
            return f'{value:.2f}'   # 2 decimals for small numbers
        else:
            return f'{value:,.1f}'  # 1 decimal for medium numbers
    return str(value)

# Apply formatting to all values - FIXED: use map() instead of applymap()
annot_formatted = annot_data.map(format_with_commas)

# ============================================================================
# STEP 5: Create Heatmap with Formatted Annotations
# ============================================================================

n_countries = len(outlier_df)
fig_height = max(8, n_countries * 0.7 + 2)

fig, ax = plt.subplots(figsize=(20, fig_height))

# Create heatmap WITHOUT annotations first
sns.heatmap(
    heatmap_data,
    annot=False,
    fmt='.1f',
    cmap='RdYlGn_r',
    center=0,
    linewidths=2,
    linecolor='white',
    cbar_kws={'label': 'Z-Score', 'shrink': 0.8},
    xticklabels=outlier_features,
    yticklabels=outlier_df['Country'].values,
    ax=ax
)

# Add custom annotations with comma formatting
for i in range(len(outlier_df)):
    for j in range(len(outlier_features)):
        value = annot_formatted.iloc[i, j]
        ax.text(
            j + 0.5, 
            i + 0.5, 
            value, 
            ha='center', 
            va='center',
            fontsize=16,
            fontweight='bold',
            color='black'
        )

ax.set_title(
    'Outlier Countries Profile\n(Red = High values, Green = Low values)', 
    fontsize=26, fontweight='bold', pad=25
)

plt.xticks(
    rotation=45, 
    ha='right', 
    fontsize=18, 
    fontweight='bold'
)

plt.yticks(
    fontsize=14,
    fontweight='bold'
)

ax.set_xlabel('Features', fontsize=20, fontweight='bold')
ax.set_ylabel('Countries', fontsize=20, fontweight='bold')

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=16)
cbar.set_label('Z-Score', size=18, weight='bold')

ax.grid(False)

plt.tight_layout()
plt.savefig(
    '../figures/eda/outlier_countries_heatmap.png', 
    dpi=300, 
    bbox_inches='tight', 
    facecolor='white', 
    edgecolor='none'
)
plt.show()

print("\n✅ Heatmap saved to: ../figures/eda/outlier_countries_heatmap.png")

In [ ]:
# ============================================================================
# 3.5 Skewness and Kurtosis Analysis
# ============================================================================

fig, axes = plt.subplots(2, 1, figsize=(18, 14)) 

skewness = df_raw[selected_features].skew().sort_values()
colors_skew = ['#C0392B' if x > 2 else '#E67E22' if x > 1 else '#2ECC71' for x in skewness.values]

bars1 = axes[0].barh(skewness.index, skewness.values, color=colors_skew, alpha=0.8)
axes[0].axvline(x=1, color='orange', linestyle='--', linewidth=2.5, label='Moderate skew (>1)')  
axes[0].axvline(x=2, color='red', linestyle='--', linewidth=2.5, label='High skew (>2)')  

axes[0].set_xlabel('Skewness', fontsize=20, fontweight='medium') 
axes[0].set_title('Feature Skewness - Higher Values Indicate Extreme Outliers', 
                 fontsize=24, fontweight='bold') 
axes[0].grid(True, alpha=0.2, axis='x')
axes[0].tick_params(axis='both', labelsize=18)  
axes[0].legend(loc='lower right', fontsize=18) 

for i, (bar, value) in enumerate(zip(bars1, skewness.values)):
    axes[0].text(value + 0.05, bar.get_y() + bar.get_height()/2, 
                f'{value:.2f}', va='center', fontsize=16, fontweight='bold')  

kurtosis = df_raw[selected_features].kurtosis().sort_values()
colors_kurt = ['#C0392B' if x > 10 else '#E67E22' if x > 5 else '#2ECC71' for x in kurtosis.values]

bars2 = axes[1].barh(kurtosis.index, kurtosis.values, color=colors_kurt, alpha=0.8)
axes[1].axvline(x=3, color='orange', linestyle='--', linewidth=2.5, label='Heavy tails (>3)') 
axes[1].axvline(x=10, color='red', linestyle='--', linewidth=2.5, label='Extreme tails (>10)') 

axes[1].set_xlabel('Kurtosis', fontsize=20, fontweight='medium')  
axes[1].set_title('Feature Kurtosis - Higher Values Indicate Heavy-tailed Distributions with Outliers', 
                 fontsize=24, fontweight='bold') 
axes[1].grid(True, alpha=0.2, axis='x')
axes[1].tick_params(axis='both', labelsize=18)  
axes[1].legend(loc='lower right', fontsize=18)  

for i, (bar, value) in enumerate(zip(bars2, kurtosis.values)):
    # Use dynamic offset for kurtosis values 
    offset = 0.5 if abs(value) < 10 else abs(value) * 0.05
    axes[1].text(value + offset, bar.get_y() + bar.get_height()/2, 
                f'{value:.2f}', va='center', fontsize=16, fontweight='bold')  

plt.suptitle('Distribution Shape Analysis - Skewness & Kurtosis', 
            fontsize=26, fontweight='bold', y=1.02)  

plt.tight_layout()
plt.savefig('../figures/eda/skewness_kurtosis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# 3.6 Transformation Impact Analysis 
# ============================================================================

print("\n📊 TRANSFORMATION IMPACT ANALYSIS")
print("=" * 60)


# Apply transformation
pt = PowerTransformer(method='yeo-johnson')
df_transformed_temp = pd.DataFrame(
    pt.fit_transform(df_raw[selected_features]),
    columns=selected_features,
    index=df_raw.index
)

# Compare skewness
skew_df = pd.DataFrame({
    'Original': df_raw[selected_features].skew(),
    'Transformed': df_transformed_temp.skew()
})

# Calculate improvement
skew_df['Improvement'] = abs(skew_df['Original']) - abs(skew_df['Transformed'])
skew_df['Improvement_Pct'] = (skew_df['Improvement'] / abs(skew_df['Original'])) * 100

print("\n📊 SKEWNESS IMPROVEMENT:")
print(skew_df.round(2))

# ============================================================================
# VISUALIZATION 
# ============================================================================

fig, ax = plt.subplots(figsize=(18, 10))  

# Sort by improvement
skew_df_sorted = skew_df.sort_values('Improvement', ascending=False)

# Create bars 
bars = ax.barh(skew_df_sorted.index, skew_df_sorted['Improvement_Pct'], 
               color=['#2ECC71' if x > 70 else '#F39C12' if x > 40 else '#E74C3C' 
                      for x in skew_df_sorted['Improvement_Pct']],
               edgecolor='white', linewidth=2)

# Add reference lines
ax.axvline(x=70, color='green', linestyle='--', linewidth=2.5, label='Excellent (>70%)')
ax.axvline(x=40, color='orange', linestyle='--', linewidth=2.5, label='Good (>40%)')

# ====== CUSTOMIZE AXES ======
ax.set_xlabel('Improvement Percentage (%)', fontsize=20, fontweight='medium')
ax.set_title('Feature Skewness Reduction After Transformation', 
             fontsize=24, fontweight='bold')
ax.tick_params(axis='both', labelsize=18)
ax.grid(True, alpha=0.2, axis='x')

ax.legend(loc='lower left', fontsize=18, framealpha=0.9, 
          edgecolor='#BDC3C7', bbox_to_anchor=(0.02, 0.02))

# Add percentage labels
for bar, pct in zip(bars, skew_df_sorted['Improvement_Pct']):
    ax.text(pct + 1, bar.get_y() + bar.get_height()/2, 
            f'{pct:.1f}%', va='center', fontsize=16, fontweight='bold')

ax.set_axisbelow(True)
ax.set_facecolor('#F8F9FA')

plt.tight_layout()
plt.savefig('../figures/eda/transformation_improvement_chart.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📈 TRANSFORMATION SUMMARY:")
print("-" * 60)
print(f"• Best improved feature: {skew_df_sorted.index[0]} ({skew_df_sorted['Improvement_Pct'].iloc[0]:.1f}%)")
print(f"• Worst improved feature: {skew_df_sorted.index[-1]} ({skew_df_sorted['Improvement_Pct'].iloc[-1]:.1f}%)")
print(f"• Average improvement: {skew_df_sorted['Improvement_Pct'].mean():.1f}%")

In [ ]:
# ============================================================================
# 4. DATA PREPROCESSING WITH TRANSFORMATION 
# ============================================================================

print("🔄 DATA PREPROCESSING")
print("=" * 60)

# Check for missing values
missing_counts = df_raw.isnull().sum()
total_missing = missing_counts.sum()
missing_per_column = (missing_counts / len(df_raw) * 100).round(2)

print(f"Total missing values: {total_missing}")
print(f"Missing values per column:\n{missing_per_column[missing_per_column > 0]}")

# Determine handling strategy based on missing data
if total_missing == 0:
    print("\n✅ No missing values detected - proceeding with clean data")
    df_clean = df_raw.copy()
elif (missing_per_column <= 5).all():
    print("\n📊 Missing values are less than 5% in all columns")
    print("   → Using listwise deletion (dropping rows with missing values)")
    rows_before = len(df_raw)
    df_clean = df_raw.dropna()
    rows_removed = rows_before - len(df_clean)
    print(f"   → Removed {rows_removed} rows ({rows_removed/rows_before*100:.1f}%)")
else:
    print("\n⚠️ Some columns have > 5% missing values:")
    high_missing = missing_per_column[missing_per_column > 5]
    for col, pct in high_missing.items():
        print(f"   • {col}: {pct:.1f}% missing")
    
    cols_to_drop = missing_per_column[missing_per_column > 20].index.tolist()
    if cols_to_drop:
        print(f"\n   → Dropping columns with > 20% missing: {cols_to_drop}")
        df_clean = df_raw.drop(columns=cols_to_drop)
    else:
        df_clean = df_raw.copy()
    
    print("\n   → For remaining missing values (< 20%), using median imputation")
    print("   → Justification: Missing values appear random and median is robust to outliers")
    
    
    imputer = SimpleImputer(strategy='median')
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        imputed_array = imputer.fit_transform(df_clean[numeric_cols])
        df_clean[numeric_cols] = imputed_array
        print(f"   → Imputed missing values in numeric columns")

# ============================================================================
# Apply Yeo-Johnson Transformation
# ============================================================================

print("\n🔄 APPLYING YEO-JOHNSON TRANSFORMATION")
print("-" * 60)



# Select features to transform
selected_features = ['child_mort', 'exports', 'health', 'imports', 'income', 
                     'inflation', 'life_expec', 'total_fer', 'gdpp']

# Store original skewness for comparison
original_skew = df_clean[selected_features].skew()

# Apply Yeo-Johnson transformation
print("   Applying Yeo-Johnson transformation to handle skewness...")
pt = PowerTransformer(method='yeo-johnson')
df_transformed = pd.DataFrame(
    pt.fit_transform(df_clean[selected_features]),
    columns=selected_features,
    index=df_clean.index
)

# Check skewness after transformation
transformed_skew = df_transformed.skew()

# Print improvement
print("\n📊 SKEWNESS IMPROVEMENT:")
print("-" * 70)
print(f"{'Feature':<15} {'Original':<12} {'Transformed':<14} {'Improvement':<12} {'Status':<10}")
print("-" * 70)

for feature in selected_features:
    orig = original_skew[feature]
    trans = transformed_skew[feature]
    improvement = abs(orig) - abs(trans)
    pct_improvement = (improvement / abs(orig)) * 100 if orig != 0 else 0
    
    if abs(trans) < 0.5:
        status = '✅ Excellent'
    elif abs(trans) < 1.0:
        status = '✅ Good'
    else:
        status = '⚠️ Moderate'
    
    print(f"{feature:<15} {orig:<12.2f} {trans:<14.2f} {improvement:<12.2f} {status:<10}")

# ============================================================================
# Standardize features
# ============================================================================

print("\n📊 STANDARDIZING FEATURES")
print("-" * 60)

# Standardize to ensure all features have equal weight
scaler = StandardScaler()
df_final = pd.DataFrame(
    scaler.fit_transform(df_transformed),
    columns=selected_features,
    index=df_clean.index
)

print("   Features standardized with mean=0, std=1")
print("\n   ✅ Outliers retained (no capping applied) - preserving natural variation")

# ============================================================================
# VISUALIZATION: Before vs After Transformation 
# ============================================================================


print("\n📊 CREATING BEFORE VS AFTER TRANSFORMATION VISUALIZATION")
print("=" * 60)

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

for i, feature in enumerate(selected_features):
    ax = axes[i]
    
    # Original distribution (blue)
    ax.hist(df_clean[feature], bins=30, alpha=0.6, color='#3498DB', 
            edgecolor='white', linewidth=1.5, label='Original', density=True)
    
    # Transformed distribution (red)
    ax.hist(df_final[feature], bins=30, alpha=0.6, color='#E74C3C', 
            edgecolor='white', linewidth=1.5, label='Transformed', density=True)
    
    # Normal distribution reference (dashed line)
    x = np.linspace(df_final[feature].min(), df_final[feature].max(), 100)
    ax.plot(x, stats.norm.pdf(x, 0, 1), 'k--', linewidth=2.5, 
            alpha=0.5, label='Normal')
    
    # Skewness and Improvement 
    skew_original = df_clean[feature].skew()
    skew_transformed = df_final[feature].skew()
    improvement = (abs(skew_original) - abs(skew_transformed)) / abs(skew_original) * 100 if skew_original != 0 else 0
    color = '#27AE60' if improvement > 90 else '#F39C12' if improvement > 70 else '#E74C3C'
    
    # Combined text box at bottom right
    info_text = f"Skew: {skew_original:.2f} → {skew_transformed:.2f}\nImprovement: {improvement:.0f}%"
    ax.text(0.95, 0.05, info_text,
            transform=ax.transAxes,
            ha='right', va='bottom',
            fontsize=14, fontweight='bold',
            color=color,
            bbox=dict(boxstyle="round,pad=0.5", facecolor='white', alpha=0.9, edgecolor='#BDC3C7'))
    
    # Labels and title
    ax.set_title(feature.replace('_', ' ').title(), fontsize=20, fontweight='bold')
    ax.set_xlabel('Value', fontsize=16, fontweight='medium')
    ax.set_ylabel('Density', fontsize=16, fontweight='medium')
    
    # Legend 
    ax.legend(loc='upper right', fontsize=14, framealpha=0.9, edgecolor='#BDC3C7')
    
    ax.grid(True, alpha=0.2)
    ax.tick_params(axis='both', labelsize=14)
    ax.set_axisbelow(True)
    ax.set_facecolor('#F8F9FA')
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)

plt.suptitle('Distribution Comparison: Original vs Transformed (Yeo-Johnson + Scaling)', 
            fontsize=28, fontweight='bold', y=1.02, color='#2C3E50')

plt.tight_layout()
plt.savefig('../figures/eda/before_after_transformation.png', 
            dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print("\n✅ Visualization saved to: ../figures/eda/before_after_transformation.png")
    



# ============================================================================
# Save processed data 
# ============================================================================

print(f"\n📊 PREPROCESSING RESULTS:")
print("-" * 60)
print(f"Original data shape: {df_raw.shape}")
print(f"Processed data shape: {df_final.shape}")
print(f"Features retained: {len(df_final.columns)}")
print(f"Feature names: {', '.join(df_final.columns.tolist())}")

print("\n🔍 PROCESSED DATA SAMPLE (first 5 rows):")
display(df_final.head())

# Save processed data
os.makedirs('../data/processed', exist_ok=True)
df_final.to_csv('../data/processed/countries_transformed_scaled.csv', index=False)
print("\n✅ Processed data saved to: ../data/processed/countries_transformed_scaled.csv")

# Save the transformer and scaler for later use
import joblib
joblib.dump(pt, '../data/processed/yeojohnson_transformer.pkl')
joblib.dump(scaler, '../data/processed/standard_scaler.pkl')
print("✅ Transformer and scaler saved for future use")

print("\n🎯 Clustering data ready! Proceed to clustering analysis.")